# Laptop Market Analysis - Data Processing Pipeline

## 1. Methodology & Execution

This pipeline cleans nested JSON data from multiple retailers into a structured, BI-ready format constraint for analysis.

**Core Processing Steps:**
- **Text Normalization:** Standardizes multi-language specs and terminology.
- **Regex Extraction Heuristics:** Employs negative lookaheads to extract hardware limitations accurately.
- **Strict Cross-Retailer Imputation:** Intelligently backfills missing data by cross-checking matching hardware nodes (`Brand`, `Series`, `CPU`, `GPU`, `Price Seg`, `Screen`). Retains missingness where inference is unsafe.
- **Categorical Binning:** Derives broad groupings (`Category`, `Price Tier`, `Hardware Generations`) based on overlapping features.

## 2. Detailed Data Schema

Output produces explicit, clean features optimized strictly for analytics platforms (Tableau / Power BI). Below are the precise data contents and constraints per field.

### Metadata & Identifiers
- **`id` / `source` / `url`**: Unique string IDs, Retailer Names (`An Phat PC`, `Gear VN`, etc.), and Direct product links.
- **`brand`**: `Asus`, `Acer`, `Dell`, `HP`, `Lenovo`, `MSI`, `Apple`, `Gigabyte`, `Other`
- **`name`**: Normalized product title string.
- **`category`**: Inferred use-case -> `Gaming`, `Creator`, `Office`
- **`price`**: Continuous integer (e.g., `15000000`).
- **`price_segment`**: `Budget` (<15M), `Mainstream` (15-25M), `Mid-high` (25-45M), `Premium` (>45M)
- **`is_available`**: Boolean `True` / `False`.

### Core Computing (CPU & GPU)
- **`is_ai`**: Boolean flag `True` / `False` denoting NPU/AI-chip presence.
- **`cpu_brand`**: `Intel`, `AMD`, `Apple`, `Qualcomm`, `Other`
- **`cpu_family`**: `Intel Core Ultra`, `Intel Core`, `AMD Ryzen AI`, `AMD Ryzen`, `Apple M-Series`, `Snapdragon (ARM)`, `Other`
- **`gpu_type`**: `Dedicated` (RTX/RX), `Integrated` (Iris/UHD), `Unified` (Apple)
- **`gpu_model`**: `Nvidia RTX/GTX`, `Workstation`, `AMD Radeon`, `Intel Arc / Integrated`, `Apple GPU`, `Other`

### Memory & Storage
- **`ram_gb`**: Integer values -> `8`, `16`, `24`, `32`, `48`, `64` *(Missing fields securely kept as `NaN`)*
- **`ram_type`**: `DDR5 / LPDDR5`, `DDR4 / LPDDR4`, `Unified`, `Other`
- **`storage_gb`**: Integer values -> `256`, `512`, `1024`, `2048` *(Missing fields securely kept as `NaN`)*
- **`storage_type`**: `SSD`, `HDD`, `eMMC`

### Display & Physical Specs
- **`screen_size_inches`**: Boundaries -> `13" - 14.5"`, `15.6" - 16.0"`, `17.3" - 18.0"`, `Other`
- **`screen_hz`**: Groupings -> `60 - 90`, `120 - 165`, `240+`, `Other`
- **`screen_res`**: `FHD / FHD+`, `2K - 3K`, `4K`, `Other`
- **`os_family`**: `Windows`, `MacOS`, `Other`
- **`weight_kg`**: Native continuous float (e.g., `1.85`).
- **`battery_wh`**: Native continuous float (e.g., `50.0`). 

### Social / Reviews
- **`avg_rating`**: User rating float (0.0 to 5.0).
- **`review_count`**: Integer reflecting total count of reviews.
- **`satisfied_count`**: Extracted aggregate integer from platform satisfaction metrics.

## Data Loading

In [1]:
import json, os, re, statistics
import pandas as pd
import unicodedata

DATA_DIR = "data"

def load_json_files(data_dir):
    all_data = []
    f_count = 0
    for file in os.listdir(data_dir):
        if file.startswith('data_') and file.endswith('.json'):
            path = os.path.join(data_dir, file)
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    if isinstance(data, list):
                        all_data.extend(data)
                    else:
                        all_data.append(data)
                f_count += 1
            except Exception as e:
                print(f"Err {file}: {e}")
    
    print(f"Loaded: {len(all_data)} records, {f_count} files")
    return all_data

raw_records = load_json_files(DATA_DIR)

Loaded: 3732 records, 7 files


## Text Cleaners & Regex 

In [2]:
def remove_accents(input_str):
    if input_str is None:
        return ""
    # Strip linguistic accents and convert to lowercase
    s1 = unicodedata.normalize('NFD', input_str)
    s2 = "".join([c for c in s1 if not unicodedata.combining(c)])
    return s2.replace('đ', 'd').replace('Đ', 'D').lower().strip()

def normalize_key(key):
    return remove_accents(key)

In [3]:
def extract_ram_capacity(ram_string):
    if not ram_string: return None
    s = str(ram_string).lower()

    for ex in ['tối đa', 'toi da', 'up to', 'nâng cấp', 'nang cap', 'max', 'còn trống', 'khe trống', 'mở rộng', 'cắm sẵn', 'nâng']:
        if ex in s: s = s.split(ex)[0]

    s_outside = re.sub(r'\(.*?\)', '', s)
    
    def get_sum(text):
        match_mul1 = re.search(r'(\d+)\s*x\s*(\d+)\s*(?:gb|g|gib)\b', text)
        if match_mul1: return int(match_mul1.group(1)) * int(match_mul1.group(2))
        
        match_mul2 = re.search(r'(\d+)\s*(?:gb|g|gib)\s*x\s*(\d+)', text)
        if match_mul2: return int(match_mul2.group(1)) * int(match_mul2.group(2))
        
        chunks = re.finditer(r'(\d+)\s*(?:gb|g|gib)\b', text)
        res = [int(m.group(1)) for m in chunks]
        return float(sum(res)) if res else None

    outside_sum = get_sum(s_outside)
    if outside_sum: return outside_sum
    
    return get_sum(s)

def extract_ram_type(ram_string):
    if not ram_string: return None
    ram_string = str(ram_string).upper()
    if 'LPDDR5' in ram_string: return 'LPDDR5/5X' # Group 5 and 5X
    if 'DDR5' in ram_string: return 'DDR5'
    if 'LPDDR4' in ram_string: return 'LPDDR4/4X' # Group 4 and 4X
    if 'DDR4' in ram_string: return 'DDR4'
    if 'UNIFIED' in ram_string or 'M-SERIES' in ram_string: return 'Unified'
    return None

def extract_storage_capacity(storage_string):
    if not storage_string: return None
    s = str(storage_string).lower()

    for ex in ['tối đa', 'toi da', 'up to', 'nâng cấp', 'nang cap', 'max', 'còn trống', 'khe trống', 'mở rộng']:
        if ex in s: s = s.split(ex)[0]
    
    s = s.replace('1tb', '1024gb').replace('1 tb', '1024gb')
    s = s.replace('2tb', '2048gb').replace('2 tb', '2048gb')
    s = s.replace('4tb', '4096gb').replace('4 tb', '4096gb')

    s_outside = re.sub(r'\(.*?\)', '', s)
    
    def get_sum(text):
        match_mul1 = re.search(r'(\d+)\s*x\s*(\d+)\s*(?:gb|g|gib)\b', text)
        if match_mul1: return int(match_mul1.group(1)) * int(match_mul1.group(2))
        match_mul2 = re.search(r'(\d+)\s*(?:gb|g|gib)\s*x\s*(\d+)', text)
        if match_mul2: return int(match_mul2.group(1)) * int(match_mul2.group(2))
        chunks = re.finditer(r'(\d+)\s*(?:gb|g|gib)\b', text)
        res = [int(m.group(1)) for m in chunks]
        return float(sum(res)) if res else None
    
    out_sum = get_sum(s_outside)
    if out_sum: return out_sum
    return get_sum(s)

def extract_storage_type(storage_string):
    # Default vector mapping
    if not storage_string: return 'SSD' 
    storage_string = str(storage_string).upper()
    has_ssd = 'SSD' in storage_string or 'NVME' in storage_string
    has_hdd = 'HDD' in storage_string
    
    if has_ssd and has_hdd: return 'Other'
    if has_ssd: return 'SSD'
    if has_hdd: return 'HDD'
    if 'EMMC' in storage_string: return 'eMMC'
    return 'SSD'

def extract_screen_size(screen_string):
    if not screen_string: return None
    val = None
    match = re.search(r'(\d{2}(?:\.\d+)?)\s*(?:\"|inch|\'\')', str(screen_string), re.IGNORECASE)
    if match:
        val = float(match.group(1))
    else:
        match_start = re.search(r'^(\d{2}(?:\.\d+)?)', str(screen_string))
        if match_start:
            val = float(match_start.group(1))
            
    if val is not None:
        if val <= 13.6: return 13.3
        if val <= 14.2: return 14.0 # Fits standard 14.0 and 14.2 panels
        if val <= 14.6: return 14.5 # Preserve the 14.5" niche (common in creator/office OLED)
        if val <= 15.6: return 15.6 # Preserves standard 15.3, 15.6
        if val <= 16.2: return 16.0 # standard 16.0 panels
        if val <= 17.3: return 17.3
        return 18.0 # standard 18-inch gaming mapped
    return None

def extract_screen_hz(screen_string):
    if not screen_string: return None
    match = re.search(r'(\d+)\s*(?:hz)', str(screen_string), re.IGNORECASE)
    
    val = None
    if match: val = int(match.group(1))
    
    if val is not None:
        if val <= 60: return 60
        if val <= 90: return 90
        if val <= 120: return 120
        if val <= 144: return 144
        if val <= 165: return 165
        if val <= 240: return 240 # Standard high esport
        if val <= 360: return 360 # Extreme esport 360
        return 480 # Cap extreme 480Hz
    return None

def extract_screen_resolution(screen_string):
    if not screen_string: return None
    screen_string = str(screen_string).upper()
    if '4K' in screen_string or '3840' in screen_string or 'UHD' in screen_string: return '4K'
    if '3K' in screen_string or '2880' in screen_string or '3024' in screen_string or '3456' in screen_string: return '2.8K - 3K'
    if '2.8K' in screen_string or '2880X1800' in screen_string or '2880 X 1800' in screen_string: return '2.8K - 3K'
    if '2.5K' in screen_string or 'WQXGA' in screen_string or '2560' in screen_string or '2K' in screen_string or 'QHD' in screen_string: return '2K - 2.5K'       
    if 'FHD+' in screen_string or '1920 X 1200' in screen_string or '1920X1200' in screen_string or 'WUXGA' in screen_string: return 'FHD+'
    if 'FHD' in screen_string or 'FULL HD' in screen_string or '1920' in screen_string: return 'FHD'
    if re.search(r'\bHD\b', screen_string): return 'FHD' # Group HD into FHD because standalone HD is statistically irrelevant now
    return None

def extract_gpu_type(gpu_string):
    if not gpu_string: return "Integrated"
    gpu_str_lower = str(gpu_string).lower()
    
    if re.search(r'\b(?:rtx|gtx|rx|nvidia|nvdia|geforce|mx\d|quadro)', gpu_str_lower): return "Dedicated"
    if re.search(r'\b(?:apple|unified|macbook)\b', gpu_str_lower): return "Unified"
    return "Integrated"

def extract_gpu_model(gpu_raw):
    if not gpu_raw: return "Other"
    
    g_str = str(gpu_raw).upper().replace('™', '').replace('®', '')
    
    # Raw exact model extraction via Regex matches
    rtx_gtx_match = re.search(r'(RTX|GTX)\s*(\d{4}(?:\s*TI|\s*SUPER)?)', g_str)
    if rtx_gtx_match:
        return f"{rtx_gtx_match.group(1)} {rtx_gtx_match.group(2).replace(' ', '')}"
        
    if 'QUADRO' in g_str:
        q_match = re.search(r'QUADRO\s*([A-Z]*\d{3,4})', g_str)
        if q_match: return f"Nvidia Quadro {q_match.group(1)}"
        return "Nvidia Quadro"
    
    if 'ARC' in g_str: 
        arc_match = re.search(r'ARC\s*(A\d{3}M|A\d{2}M|[A-Z0-9]+)', g_str)
        if arc_match: 
            return f"Intel Arc GRAPHICS" if arc_match.group(1) == "GRAPHICS" else f"Intel Arc {arc_match.group(1)}"
        return "Intel Arc GRAPHICS"
        
    if 'IRIS' in g_str or 'X\\' in g_str or 'XE' in g_str: return "Intel Iris Xe"
    if 'UHD' in g_str or 'HD GRAPHICS' in g_str: return "Intel UHD Graphics"
    
    rx_match = re.search(r'RX\s*(\d{4}(?:M|S|XT)?)', g_str)
    if rx_match: return f"Radeon RX {rx_match.group(1)}"
    if 'RADEON' in g_str: return "AMD Radeon"
    
    app_match = re.search(r'\b(M[1-5](?:\s*PRO|\s*MAX|\s*ULTRA)?)\b', g_str)
    if app_match: return f"Apple {app_match.group(1).title()} GPU"
    if 'APPLE' in g_str or 'NHÂN GPU' in g_str or 'CORE GPU' in g_str: 
        return "Apple GPU"
        
    adreno = re.search(r'ADRENO\s*(\d+)', g_str)
    if adreno: return f"Qualcomm Adreno {adreno.group(1)}"
    if 'SNAPDRAGON' in g_str: return "Qualcomm Adreno"
    
    mx_match = re.search(r'MX\d{3}', g_str)
    if mx_match: return f"GeForce {mx_match.group(0)}"

    return "Intel Integrated"

def extract_os_family(os_raw):
    if not os_raw: return "Windows 11" 
    os_str = str(os_raw).lower()
    if 'mac' in os_str: return "macOS"
    if 'windows 10' in os_str or 'win 10' in os_str: return "Windows 10"
    if 'dos' in os_str or 'linux' in os_str or 'ubuntu' in os_str: return "Linux/DOS"
    # Lumping "Windows" general strings or Win 11 into Windows 11 as it's the current default
    return "Windows 11" 

def extract_weight_kg(weight_raw):
    if not weight_raw: return None
    weight_str = str(weight_raw).lower()
    
    # Grammatical pattern validation and float conversion mapping
    mg = re.search(r'(?<!k)(?<!k\s)(?<!\.)(\d{3,4})\s*(?:g|gram)\b', weight_str)
    if mg:
        val = int(mg.group(1))
        if 500 <= val <= 5000:
            return round(val / 1000.0, 2)
            
    match = re.search(r'(\d+[\.,]\d+|\d+)\s*(?:kg)', weight_str, re.IGNORECASE)
    if match:
        val = match.group(1).replace(',', '.')
        try:
            return float(val)
        except ValueError:
            return None
    
    m2 = re.search(r'(?:nặng|lượng|weight|khối)[\s\:\-]*(\d+[\.,]\d+|\d+)', weight_str)
    if m2:
        val = m2.group(1).replace(',', '.')
        try:
            return float(val)
        except ValueError:
            pass
    return None

def extract_battery_wh(battery_raw):
    if not battery_raw: return None
    s = str(battery_raw).lower()
    
    # Extract explicitly formatted Wh or Whr (e.g., 50 Wh, 42.5 whr, 90 w.h)
    match = re.search(r'(\d+(?:[\.,]\d+)?)\s*(?:wh|whr|w\.h|whrs)\b', s)
    if match:
        val = match.group(1).replace(',', '.')
        try:
            return float(val)
        except ValueError:
            pass
            
    return None

def extract_cpu_family(cpu_raw):
    if not cpu_raw: return "Other"
    cpu_str = str(cpu_raw).lower().replace('™', '').replace('®', '')

    # Raw extracted labels directly from regex parsing
    apple_match = re.search(r'\b(m[1-5](?:\s*pro|\s*max|\s*ultra)?)\b', cpu_str)
    if apple_match: return f"Apple {apple_match.group(1).title()}"

    ultra_match = re.search(r'(?:core\s*)?ultra\s*(\d)', cpu_str)
    if ultra_match: return f"Core Ultra {ultra_match.group(1)}"
    
    ultral_match = re.search(r'ultral\s*(\d)', cpu_str)
    if ultral_match: return f"Core Ultra {ultral_match.group(1)}"

    core_match = re.search(r'core\s*(?:i)?\s*(3|5|7|9)', cpu_str)
    if core_match: 
        return f"Core i{core_match.group(1)}"
    else:
        i_match = re.search(r'\b(i[3579])\b', cpu_str)
        if i_match:
            return f"Core {i_match.group(1)}"
    
    if 'pentium' in cpu_str: return "Intel Pentium"
    if 'celeron' in cpu_str: return "Intel Celeron"
    if 'n100' in cpu_str: return "Intel N100"
    if 'n200' in cpu_str: return "Intel N200"

    ryzen_ai_match = re.search(r'ryzen\s*a[il]\s*(?:max\+\s*)?(?:\d+)?', cpu_str)
    if ryzen_ai_match: return "Ryzen AI"

    ryzen_match = re.search(r'ryzen\s*(?:z?\d)', cpu_str)
    if ryzen_match:
        match_str = ryzen_match.group(0).title()
        return match_str.replace("Z1", "Z1")
    else:
        r_match = re.search(r'\b(r[3579])\b', cpu_str)
        if r_match:
            return f"Ryzen {r_match.group(1).replace('r', '')}"

    if 'snapdragon' in cpu_str: 
        snap = re.search(r'snapdragon\s*(x\s*plus|x\s*elite|7c|8cx)', cpu_str)
        if snap: return f"Snapdragon {snap.group(1).title()}"
        return "Snapdragon"
        
    return "Other"

def get_price_segment(price):
    if price is None or price == 0: return None
    if price < 15000000: return "Budget"
    if price <= 25000000: return "Mainstream"
    if price <= 45000000: return "Mid-high"
    return "Premium"

def get_category(name, gpu_type, screen_hz, ram_gb):
    name_lower = str(name).lower() if name else ""

    gaming_keywords = ['gaming', 'nitro', 'tuf', 'rog ', 'legion', 'alienware', 'predator', 'loq', 'ideapad gaming', 'victus', 'cyborg', 'katana', 'bravo', 'stealth', 'raider', 'omen', 'tuf dash']
    creator_keywords = ['creator', 'studio', 'proart', 'conceptd', 'zenbook pro', 'aero', 'macbook pro']
    office_keywords = ['vivobook', 'zenbook', 'expertbook', 'swift', 'aspire', 'ideapad', 'thinkpad', 'thinkbook', 'macbook air', 'elitebook', 'probook', 'pavilion', 'envy', 'hp 14', 'hp 15']    

    # Creator tier hierarchy
    if any(kw in name_lower for kw in creator_keywords): return "Creator"
    
    # Predefined keyword matching arrays
    if any(kw in name_lower for kw in gaming_keywords): return "Gaming"
    if any(kw in name_lower for kw in office_keywords): return "Office"

    # Hardware component inference models
    if ram_gb and ram_gb >= 32 and gpu_type == "Dedicated": return "Creator"
    if gpu_type == "Dedicated" and screen_hz and screen_hz >= 120: return "Gaming"

    return "Office"

def check_ai_chip(cpu_raw, name):
    combine_str = f"{cpu_raw or ''} {name or ''}".lower()
    ai_keywords = ['npu', 'ai boost', 'ultra', 'ryzen ai', 'snapdragon', 'copilot', 'xdna', 'neural engine']
    return any(kw in combine_str for kw in ai_keywords)

def parse_satisfied_count(text):
    if not text: return 0
    match_k = re.search(r'([\d\.,]+)\s*(k|K|triệu)?\s*(khách|lượt)', str(text).lower())
    if match_k:
        num_str = match_k.group(1).replace(',', '.')
        try:
            num = float(num_str)
        except ValueError:
            return 0
        unit = match_k.group(2)
        if unit == 'k': return int(num * 1000)
        elif unit == 'triệu': return int(num * 1000000)
        return int(num)

    match_digit = re.search(r'^\s*([\d\.,]+)', str(text))
    if match_digit:
        num_str = match_digit.group(1).replace(',', '.')
        try:
            return int(float(num_str))
        except ValueError:
            pass
    return 0

## Transformer Class

In [4]:
class LaptopTransformer:
    def __init__(self):
        # Attribute standardization map
        self.key_mapping = {
            'cpu': ['cpu', 'vi xu ly', 'xu ly', 'chip', 'bo xu ly'],
            'ram': ['ram', 'bo nho trong', 'bo nho'],
            'storage': ['o cung', 'rom', 'storage', 'ssd', 'hdd', 'luu tru', 'dung luong'],
            'vga': ['vga', 'card', 'do hoa', 'gpu'], 
            'screen': ['man hinh', 'hien thi', 'display', 'do phan giai', 'tan so', 'tam nen', 'screen'], 
            'os': ['os', 'he dieu hanh', 'operating system'],
            'weight': ['weight', 'khoi luong', 'trong luong', 'kich thuoc', 'nang', 'trong'],
            'battery': ['pin', 'battery', 'dung luong pin']
        }
        # Parameter exclusion filters 
        self.excluded_keys = ['toi da', 'max', 'nang cap', 'up to', 'phu', 'reader', 'the nho']

    def _find_spec(self, raw_specs, spec_type):
        """Aggregate specs using intersection of target and exclusion parameters"""
        target_keys = self.key_mapping.get(spec_type, [])
        found_values = []
        for k, v in raw_specs.items():
            norm_k = normalize_key(k)
            
            # Sub-filter: Exclude graphics capacity strings parsing as system memory
            if spec_type == 'ram' and any(x in norm_k for x in ['do hoa', 'card', 'vga', 'dem', 'cache', 'gpu', 'vram']):
                continue
            
            # Sub-filter: Exclude system memory constraints parsing as storage vectors
            if spec_type == 'storage' and any(x in norm_k for x in ['ram', 'pin', 'the nho', 'vga', 'card', 'do hoa', 'gpu', 'vram', 'kem theo']):
                continue
            
            # Match parameters bypassing constraint constants 
            if any(tk in norm_k for tk in target_keys) and not any(ek in norm_k for ek in self.excluded_keys):
                found_values.append(str(v).strip())
        return " | ".join(found_values)

    def process(self, raw_data):
        # Omit irrelevant payload nodes (accessories, empty entities)
        name = raw_data.get('product', {}).get('name', '')
        
        if not name or name == 'null' or "balo" in name.lower() or "chuột" in name.lower() or "túi" in name.lower() or "apple care" in name.lower():
            return None

        current_price = raw_data.get('pricing', {}).get('current_price', 0)     
        is_available = raw_data.get('product', {}).get('is_available', True)    

        if current_price == 0:
            return None

        specs_dict = raw_data.get('specs_raw', {})
        if not specs_dict:
             return None

        brand_match = re.search(r'(Asus|Acer|Dell|HP|Lenovo|MSI|Apple|Macbook|Gigabyte|LG|Microsoft|Surface)', name, re.IGNORECASE)
        if brand_match:
            b_name = brand_match.group(1).title()
            if b_name == "Macbook": brand = "Apple"
            elif b_name == "Hp": brand = "HP"
            elif b_name == "Msi": brand = "MSI"
            elif b_name == "Lg": brand = "LG"
            elif b_name == "Surface": brand = "Microsoft"
            else: brand = b_name
        else:
            brand = "Other"

        # Apply fallback node inspection via native product title matching
        raw_cpu = self._find_spec(specs_dict, 'cpu')
        if not raw_cpu: raw_cpu = name 
        
        raw_ram = self._find_spec(specs_dict, 'ram')
        if not raw_ram: raw_ram = name
        
        raw_storage = self._find_spec(specs_dict, 'storage')
        if not raw_storage: raw_storage = name
        
        raw_vga = self._find_spec(specs_dict, 'vga')
        if not raw_vga: raw_vga = name
        
        raw_screen = self._find_spec(specs_dict, 'screen')
        raw_os = self._find_spec(specs_dict, 'os')
        raw_weight = self._find_spec(specs_dict, 'weight')
        raw_battery = self._find_spec(specs_dict, 'battery')

        # Feature extraction phase
        cpu_family = extract_cpu_family(raw_cpu)
        if cpu_family == "Other" and brand == "Apple":
            m_match = re.search(r'\b(m[1-5](?:\s*pro|\s*max)?)\b', name.lower())
            if m_match:
                cpu_family = f"Apple {m_match.group(1).title()}"

        # CPU model logic mapping
        cpu_str_lower = raw_cpu.lower()
        if "intel" in cpu_str_lower or "core" in cpu_family.lower() or "pentium" in cpu_family.lower() or "celeron" in cpu_family.lower() or "n-series" in cpu_family.lower():
            cpu_brand = "Intel"
        elif "amd" in cpu_str_lower or "ryzen" in cpu_family.lower():
            cpu_brand = "AMD"
        elif "apple" in cpu_str_lower or "m1" in cpu_str_lower or "m2" in cpu_str_lower or "m3" in cpu_str_lower or "m4" in name.lower() or brand == "Apple" or "apple" in cpu_family.lower():   
            cpu_brand = "Apple"
        elif "snapdragon" in cpu_str_lower or "snapdragon" in cpu_family.lower():
            cpu_brand = "Qualcomm"
        else:
            cpu_brand = "Other"
            
        is_ai = check_ai_chip(raw_cpu + str(specs_dict), name)

        gpu_model = extract_gpu_model(raw_vga)
        if brand == "Apple" and gpu_model == "Other":
            gpu_model = "Apple GPU"

        gpu_type = extract_gpu_type(raw_vga)
        if gpu_type == "Integrated" and gpu_model != "Other":
            if any(key in gpu_model.upper() for key in ['RTX', 'GTX', ' RX ', 'QUADRO', 'ARC A']):
                gpu_type = "Dedicated"
            elif "APPLE" in gpu_model.upper():
                gpu_type = "Unified"

        ram_gb = extract_ram_capacity(raw_ram)
        ram_type = extract_ram_type(raw_ram)
        
        storage_gb = extract_storage_capacity(raw_storage)
        storage_type = extract_storage_type(raw_storage)

        # M-Series core architecture constant assignments
        if brand == "Apple": 
            if not storage_type and storage_gb: storage_type = "SSD"
            if not ram_type and "Apple M" in cpu_family: ram_type = "Unified"

        screen_size = extract_screen_size(raw_screen)
        screen_hz = extract_screen_hz(raw_screen)
        screen_res = extract_screen_resolution(raw_screen)
        
        os_family = extract_os_family(raw_os)
        if brand == "Apple": os_family = "macOS" 
        
        weight_kg = extract_weight_kg(raw_weight)
        battery_wh = extract_battery_wh(raw_battery)

        cat = get_category(name, gpu_type, screen_hz, ram_gb)

        master_json = {
            "id": raw_data.get("id"),
            "source": raw_data.get("metadata", {}).get("source"),
            "url": raw_data.get("metadata", {}).get("url"),
            
            "brand": brand,
            "name": name,
            "category": cat,
            "is_ai": is_ai,
            "is_available": is_available,
            
            "price": current_price,
            "price_segment": get_price_segment(current_price),

            "cpu_brand": cpu_brand,
            "cpu_family": cpu_family,

            "gpu_type": gpu_type,
            "gpu_model": gpu_model,
            
            "ram_gb": ram_gb,
            "ram_type": ram_type,

            "storage_gb": storage_gb,
            "storage_type": storage_type,

            "screen_size_inches": screen_size,
            "screen_res": screen_res,
            "screen_hz": screen_hz,

            "os_family": os_family,
            "weight_kg": weight_kg,
            "battery_wh": battery_wh,
            
            "avg_rating": raw_data.get("social", {}).get("avg_rating", 0.0),      
            "review_count": raw_data.get("social", {}).get("review_count", 0),        
            "satisfied_count": parse_satisfied_count(raw_data.get("social", {}).get("satisfied_info", ""))
        }

        return master_json

## ETL Data Processor

In [5]:
import pandas as pd
import json
import os
import statistics

class DataImputer:
    """Missing data interpolation engine utilizing contextual similarity metrics"""
    def __init__(self, dataset):
        self.dataset = dataset
        self.stats = {
            'weight': 0, 'hz': 0, 'ram_type': 0, 
            'ram_cap': 0, 'storage': 0, 'res': 0, 'battery': 0
        }
        
        # Initialize logical parameters
        for row in self.dataset:
            row['_series'] = self._get_series(row.get('name', ''))
            
            price_seg = row.get('price_segment')
            if price_seg is None or price_seg == "":
                row['_price_seg'] = 'Unknown'
            else:
                row['_price_seg'] = price_seg

        self.pools = {}
        
        # Mapping matrices definition arrays
        self.pools['weight_strict'] = self._build_pool('weight_kg', ['brand', '_series', 'screen_size_inches', 'gpu_type'])
        self.pools['weight_office'] = self._build_pool('weight_kg', ['brand', '_series', 'screen_size_inches'], allow_other=True)
        
        # Tighten battery pool for precision
        self.pools['battery'] = self._build_pool('battery_wh', ['brand', '_series', 'screen_size_inches', 'category'], allow_other=True)
        self.pools['battery_loose'] = self._build_pool('battery_wh', ['brand', '_series', 'screen_size_inches'], allow_other=True)
        
        self.pools['ram_type'] = self._build_pool('ram_type', ['brand', '_series', 'cpu_family'], allow_other=True)
        self.pools['ram_type_cpu'] = self._build_pool('ram_type', ['cpu_family'], allow_other=False)
        
        self.pools['res'] = self._build_pool('screen_res', ['brand', '_series', 'screen_size_inches', '_price_seg'], allow_other=True)
        self.pools['hz'] = self._build_pool('screen_hz', ['brand', '_series', 'screen_size_inches', 'category'], allow_other=True)
        
        # Strict cross-checking limits for raw component capabilities
        self.pools['ram_strict'] = self._build_pool('ram_gb', ['brand', '_series', 'cpu_family', 'gpu_model', '_price_seg', 'screen_size_inches'], allow_other=False)
        self.pools['storage_strict'] = self._build_pool('storage_gb', ['brand', '_series', 'cpu_family', 'gpu_model', '_price_seg', 'screen_size_inches'], allow_other=False)

    def _get_series(self, name):
        """Extract primary product series identifier"""
        series_list = [
            'macbook air', 'macbook pro', 'tuf', 'rog', 'zephyrus', 'vivobook', 'zenbook', 'expertbook', 
            'nitro', 'predator', 'aspire', 'swift', 'spin', 'conceptd', 'legion', 'ideapad', 'thinkpad', 
            'thinkbook', 'loq', 'yoga', 'alienware', 'inspiron', 'vostro', 'xps', 'latitude', 'precision',
            'victus', 'omen', 'pavilion', 'envy', 'spectre', 'probook', 'elitebook', 'bravo', 'katana', 
            'cyborg', 'stealth', 'raider', 'titan', 'modern', 'prestige', 'summit'
        ]
        
        name_lower = str(name).lower()
        for series in series_list:
            if series in name_lower:
                return series
        return "Unknown"

    def _build_pool(self, target_key, keys, allow_other=False):
        """Aggregate statistical mode against bounded feature combinations"""
        pools_groups = {}
        for row in self.dataset:
            val = row.get(target_key)
            if val is None or pd.isna(val):
                continue
            
            sig = tuple(row.get(k) for k in keys)
            
            if any(not v or v == "Unknown" for v in sig):
                continue
                
            if not allow_other and any(str(v) == "Other" for v in sig):
                continue
            
            if sig not in pools_groups:
                pools_groups[sig] = []
            pools_groups[sig].append(val)
            
        inferences = {}
        for sig, vals in pools_groups.items():
            try: 
                inferences[sig] = statistics.mode(vals)
            except statistics.StatisticsError: 
                inferences[sig] = vals[0]
                
        return inferences

    def _assign(self, row, key, pool_key, sig, stat_key, fallback=None):
        if not row.get(key):
            if sig in self.pools[pool_key]:
                row[key] = self.pools[pool_key][sig]
                self.stats[stat_key] += 1
            elif fallback is not None:
                row[key] = fallback
                self.stats[stat_key] += 1

    def run(self):
        for row in self.dataset:
            brand = row.get('brand')
            series = row.get('_series')
            screen_size = row.get('screen_size_inches')
            cpu = row.get('cpu_family')
            category = row.get('category')
            price_seg = row.get('_price_seg')
            gpu = row.get('gpu_model')
            gpu_type = row.get('gpu_type')
            name_str = str(row.get('name', '')).lower()
            
            self._assign(row, 'weight_kg', 'weight_strict', (brand, series, screen_size, gpu_type), 'weight')
            if not row.get('weight_kg'): 
                self._assign(row, 'weight_kg', 'weight_office', (brand, series, screen_size), 'weight')
            
            self._assign(row, 'battery_wh', 'battery', (brand, series, screen_size, category), 'battery')
            if not row.get('battery_wh'):
                self._assign(row, 'battery_wh', 'battery_loose', (brand, series, screen_size), 'battery')
            
            self._assign(row, 'screen_res', 'res', (brand, series, screen_size, price_seg), 'res')
            
            # Framework logical fallbacks implementation
            fallback_hz = None
            if category == "Office" and price_seg in ["Budget", "Mainstream"]: 
                fallback_hz = 60
            elif brand == "Apple": 
                is_pro_m_series = "pro" in name_str and any(m in name_str for m in ["m1", "m2", "m3", "m4"])
                is_large_screen = screen_size in [14.0, 14.2, 16.0, 16.2]
                fallback_hz = 120 if (is_pro_m_series and is_large_screen) else 60
            elif category == "Gaming" and price_seg == "Budget":
                fallback_hz = 144
                    
            self._assign(row, 'screen_hz', 'hz', (brand, series, screen_size, category), 'hz', fallback_hz)

            self._assign(row, 'ram_type', 'ram_type', (brand, series, cpu), 'ram_type')
            self._assign(row, 'ram_type', 'ram_type_cpu', (cpu,), 'ram_type')
            
            # Safe Cross-retailer strict matching 
            self._assign(row, 'ram_gb', 'ram_strict', (brand, series, cpu, gpu, price_seg, screen_size), 'ram_cap')
            self._assign(row, 'storage_gb', 'storage_strict', (brand, series, cpu, gpu, price_seg, screen_size), 'storage')
            
        for row in self.dataset:
            self._refine_labels(row)

        print(f"Interpolation completed. Imputed values: {self.stats}")
        return self.dataset

    def _refine_labels(self, row):
        # BRAND
        b = row.get('brand', 'Other')
        row['brand'] = b if b in ['Asus', 'Acer', 'Dell', 'HP', 'Lenovo', 'MSI', 'Apple', 'Gigabyte'] else 'Other'
            
        # OS_FAMILY
        os_f = str(row.get('os_family', '')).lower()
        if 'macos' in os_f: row['os_family'] = 'MacOS'
        elif 'windows' in os_f: row['os_family'] = 'Windows'
        else: row['os_family'] = 'Other'

        # CPU_FAMILY
        cpu = str(row.get('cpu_family', '')).lower()
        if 'ultra' in cpu: row['cpu_family'] = 'Intel Core Ultra'
        elif 'core' in cpu: row['cpu_family'] = 'Intel Core'
        elif 'ryzen ai' in cpu: row['cpu_family'] = 'AMD Ryzen AI'
        elif 'ryzen' in cpu: row['cpu_family'] = 'AMD Ryzen'
        elif 'apple' in cpu: row['cpu_family'] = 'Apple M-Series'
        elif 'snapdragon' in cpu: row['cpu_family'] = 'Snapdragon (ARM)'
        else: row['cpu_family'] = 'Other'

        # GPU_MODEL
        gpu = str(row.get('gpu_model', '')).lower()
        if 'rtx' in gpu or 'gtx' in gpu or 'geforce' in gpu: row['gpu_model'] = 'Nvidia RTX/GTX'
        elif 'quadro' in gpu or 'workstation' in gpu: row['gpu_model'] = 'Workstation'
        elif 'radeon' in gpu or 'rx' in gpu: row['gpu_model'] = 'AMD Radeon'
        elif 'intel' in gpu or 'arc' in gpu or 'iris' in gpu or 'uhd' in gpu: row['gpu_model'] = 'Intel Arc / Integrated'
        elif 'apple' in gpu: row['gpu_model'] = 'Apple GPU'
        else: row['gpu_model'] = 'Other'

        # RAM_GB
        ram = row.get('ram_gb')
        row['ram_gb'] = int(ram) if ram in [8, 16, 24, 32, 48, 64] else None

        # RAM_TYPE
        rt = str(row.get('ram_type', '')).upper()
        if '5' in rt: row['ram_type'] = 'DDR5 / LPDDR5'
        elif '4' in rt: row['ram_type'] = 'DDR4 / LPDDR4'
        elif 'UNIFIED' in rt: row['ram_type'] = 'Unified'
        else: row['ram_type'] = 'Other'

        # STORAGE_GB
        st = row.get('storage_gb')
        row['storage_gb'] = int(st) if st in [256, 512, 1024, 2048] else None

        # SCREEN_SIZE
        sz = row.get('screen_size_inches')
        if not sz: row['screen_size_inches'] = 'Other'
        elif sz <= 14.5: row['screen_size_inches'] = '13" - 14.5"'
        elif sz <= 16.2: row['screen_size_inches'] = '15.6" - 16.0"'
        elif sz <= 18.0: row['screen_size_inches'] = '17.3" - 18.0"'
        else: row['screen_size_inches'] = 'Other'

        # SCREEN_HZ
        hz = row.get('screen_hz')
        if not hz: row['screen_hz'] = 'Other'
        elif hz <= 90: row['screen_hz'] = '60 - 90'
        elif hz <= 165: row['screen_hz'] = '120 - 165'
        else: row['screen_hz'] = '240+'

        # SCREEN_RES
        res = str(row.get('screen_res', ''))
        if res in ['FHD', 'FHD+']: row['screen_res'] = 'FHD / FHD+'
        elif res in ['2K - 2.5K', '2.8K - 3K']: row['screen_res'] = '2K - 3K'
        elif res in ['4K']: row['screen_res'] = '4K'
        else: row['screen_res'] = 'Other'

        # Cleanup internal keys used by Imputer
        row.pop('_series', None)
        row.pop('_price_seg', None)

transformer = LaptopTransformer()
clean_data = [p for raw in raw_records if (p := transformer.process(raw))]

imputer = DataImputer(clean_data)
final_data = imputer.run()

print(f"Total processed records for Dashboard: {len(final_data)}")

# Export to JSON & CSV
out_path_json = os.path.join(DATA_DIR, "transformed_data.json")
with open(out_path_json, 'w', encoding='utf-8') as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

out_path_csv = os.path.join(DATA_DIR, "transformed_data.csv")
pd.DataFrame(final_data).to_csv(out_path_csv, index=False, encoding='utf-8-sig')


Interpolation completed. Imputed values: {'weight': 1040, 'hz': 1212, 'ram_type': 367, 'ram_cap': 126, 'storage': 223, 'res': 293, 'battery': 744}
Total processed records for Dashboard: 2797


## Missing Data Statistics

In [6]:
import pandas as pd

df = pd.DataFrame(final_data)
total = len(df)

missing = df.isnull().sum().to_frame('Missing_Count')
missing['Missing_Percentage'] = (missing['Missing_Count'] / total) * 100

# Filter & sort missing values
missing = missing[missing['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
missing['Missing_Percentage'] = missing['Missing_Percentage'].apply(lambda x: f"{x:.2f}%")

print(f"Data Validation across {total} records. Missing points:")
missing

Data Validation across 2797 records. Missing points:


,Missing_Count,Missing_Percentage
weight_kg,367,13.12%
storage_gb,311,11.12%
battery_wh,244,8.72%
ram_gb,242,8.65%


## Categorical Labels Distribution

In [7]:
# Verify the final category distribution for Dashboard filters
categorical_columns = [
    'brand', 'category', 'is_ai', 'price_segment', 
    'cpu_brand', 'cpu_family', 'gpu_type', 'gpu_model', 
    'ram_gb', 'ram_type', 'storage_gb', 'storage_type', 
    'screen_size_inches', 'screen_res', 'screen_hz', 'os_family',
    'source'
]

for col in categorical_columns:
    if col in df.columns:
        counts = df[col].value_counts(dropna=False)
        print(f"--- {col.upper()} ---")
        print(counts.to_string())
        print("-" * 30 + "\n")

--- BRAND ---
brand
Asus        602
Lenovo      545
Acer        383
HP          367
MSI         288
Dell        282
Apple       195
Gigabyte     80
Other        55
------------------------------

--- CATEGORY ---
category
Office     1751
Gaming      885
Creator     161
------------------------------

--- IS_AI ---
is_ai
False    1635
True     1162
------------------------------

--- PRICE_SEGMENT ---
price_segment
Mainstream    1079
Mid-high      1008
Premium        513
Budget         197
------------------------------

--- CPU_BRAND ---
cpu_brand
Intel       2038
AMD          479
Apple        183
Other         64
Qualcomm      33
------------------------------

--- CPU_FAMILY ---
cpu_family
Intel Core          1278
Intel Core Ultra     739
AMD Ryzen            299
Apple M-Series       186
AMD Ryzen AI         180
Other                 82
Snapdragon (ARM)      33
------------------------------

--- GPU_TYPE ---
gpu_type
Integrated    1673
Dedicated      991
Unified        133
---------